# Feature Union and Target Transformation Workflow

This notebook demonstrates how to use Scikit-Learn's `FeatureUnion` for combining different feature engineering pipelines and `TransformedTargetRegressor` for regression with target-side transformations (e.g., log scaling).

In [ ]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

### Loading the Dataset
Note: The data path is adjusted to point to `Train.csv` in the current directory.

In [ ]:
df = pd.read_csv("Train.csv")

print("Dataset Shape:", df.shape)
df.head()

### Preprocessing and Feature Engineering
We drop the target variable and apply one-hot encoding to categorical features.

In [ ]:
X = df.drop("Reached.on.Time_Y.N", axis=1)
y = df["Reached.on.Time_Y.N"]
X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Building the Feature Union Pipeline
We combine two pipelines:
1. **PCA Pipeline**: Standardizes features and reduces dimensionality to 2 components.
2. **Selection Pipeline**: Selects the top 2 features using univariate statistic tests (`f_classif`).

In [ ]:
pca_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2))
])

select_pipeline = Pipeline([
    ("select", SelectKBest(score_func=f_classif, k=2))
])

combined_features = FeatureUnion([
    ("pca_features", pca_pipeline),
    ("selected_features", select_pipeline)
])

X_combined = combined_features.fit_transform(X_train, y_train)
print("Combined Feature Shape:", X_combined.shape)

### Target Transformation and Regression
We use `TransformedTargetRegressor` to apply a log transformation (`np.log1p`) to the target before training and an inverse transform (`np.expm1`) when predicting.

In [ ]:
model = TransformedTargetRegressor(
    regressor=LinearRegression(),
    func=np.log1p,
    inverse_func=np.expm1
)

model.fit(X_combined, y_train)

print("Training Score:", model.score(X_combined, y_train))

predictions = model.predict(X_combined)
print("First 5 Predictions:", predictions[:5])

### Notes
- `FeatureUnion` is useful for extracting different types of features in parallel.
- `TransformedTargetRegressor` simplifies applying transformations to the target variable without manual bookkeeping.